# 在 Google Colab 编译 pissarro/pissarropro TWRP

此笔记本构建 Android 12.1 分支的 `recovery-as-boot` boot 镜像，只生成并校验产物，不会连接或刷写手机。建议先在 Colab 中选择高内存运行时，然后从上到下执行。运行时可能被回收；源码必须放在 `/content`，不要放入 Google Drive。

In [ ]:
# 检查本次 Colab 分配到的资源；不足时停止，换一个运行时再试。
import os
import shutil

disk_free = shutil.disk_usage('/content').free
with open('/proc/meminfo', encoding='utf-8') as meminfo:
    memory_kib = int(next(line for line in meminfo if line.startswith('MemTotal:')).split()[1])
memory_bytes = memory_kib * 1024
print(f'CPU：{os.cpu_count()}')
print(f'可用磁盘：{disk_free / 1024**3:.1f} GiB')
print(f'物理内存：{memory_bytes / 1024**3:.1f} GiB')
assert disk_free >= 35 * 1024**3, '可用磁盘不足 35 GiB，请重新分配运行时。'
assert memory_bytes >= 10 * 1024**3, '物理内存不足 10 GiB，请选择高内存运行时。'


In [ ]:
%%bash
set -Eeuo pipefail
sudo apt-get update
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends \
  bc bison build-essential ccache curl flex g++-multilib gcc-multilib git gnupg \
  gperf imagemagick lib32ncurses-dev lib32readline-dev lib32z1-dev libc6-dev-i386 \
  libgl1-mesa-dev liblz4-tool libncurses-dev libssl-dev libxml2-utils lzop \
  openjdk-11-jdk-headless pngcrush python3 rsync schedtool squashfs-tools unzip \
  x11proto-core-dev xsltproc zip zlib1g-dev
sudo apt-get clean

repo_version='2.54'
repo_sha256='6cba294d6218bbd4a1500598207b3979c752c7a122aef9429e4d7fef688833b5'
mkdir -p /content/repo-bin
curl --fail --location --retry 3 --silent --show-error \
  "https://storage.googleapis.com/git-repo-downloads/repo-${repo_version}" \
  --output /content/repo-bin/repo
printf '%s  %s\n' "${repo_sha256}" /content/repo-bin/repo | sha256sum --check --status
chmod 0755 /content/repo-bin/repo
java -version
/content/repo-bin/repo version


In [ ]:
%%bash
set -Eeuo pipefail
source_dir=/content/twrp-pissarro
artifact_dir=/content/twrp-artifacts
mkdir -p "${source_dir}" "${artifact_dir}"
cd "${source_dir}"
git config --global user.name 'TWRP Builder'
git config --global user.email 'twrp-builder@users.noreply.github.com'
git config --global color.ui false
/content/repo-bin/repo init \
  --depth=1 \
  --no-clone-bundle \
  --repo-url=https://github.com/GerritCodeReview/git-repo.git \
  --repo-rev=v2.54 \
  -u https://github.com/minimal-manifest-twrp/platform_manifest_twrp_aosp.git \
  -b twrp-12.1
mkdir -p .repo/local_manifests
cat > .repo/local_manifests/pissarro.xml <<'XML'
<?xml version="1.0" encoding="UTF-8"?>
<manifest>
  <remote name="github-pissarro" fetch="https://github.com/" />
  <project name="TeamWin/android_device_xiaomi_pissarro"
           path="device/xiaomi/pissarro"
           remote="github-pissarro"
           revision="962f972bca23593ed763f650977c64274b4e9f25"
           clone-depth="1" />
</manifest>
XML
set -o pipefail
/content/repo-bin/repo sync \
  --current-branch --detach --fail-fast --force-sync \
  --no-clone-bundle --no-tags --retry-fetches=3 --jobs=4 \
  2>&1 | tee "${artifact_dir}/sync.log"
/content/repo-bin/repo manifest --revision-as-HEAD > "${artifact_dir}/repo-manifest.xml"
df -h "${source_dir}" | tee "${artifact_dir}/disk-after-sync.txt"
du -sh "${source_dir}" | tee -a "${artifact_dir}/disk-after-sync.txt"
available_kib="$(df -Pk "${source_dir}" | awk 'NR == 2 {print $4}')"
(( available_kib >= 10 * 1024 * 1024 )) || { echo '同步后可用磁盘不足 10 GiB。'; exit 1; }


In [ ]:
%%bash
set -Eeuo pipefail
source_dir=/content/twrp-pissarro
artifact_dir=/content/twrp-artifacts
board_config="${source_dir}/device/xiaomi/pissarro/BoardConfig.mk"
grep -Eq '^BOARD_USES_RECOVERY_AS_BOOT[[:space:]]*:=[[:space:]]*true' "${board_config}"
grep -Eq '^TARGET_NO_RECOVERY[[:space:]]*:=[[:space:]]*true' "${board_config}"
grep -Eq '^TW_NO_FASTBOOT_BOOT[[:space:]]*:=[[:space:]]*true' "${board_config}"
memory_kib="$(awk '/^MemTotal:/ {print $2}' /proc/meminfo)"
build_jobs=2
(( memory_kib >= 20 * 1024 * 1024 )) && build_jobs=4
echo "编译并发数：${build_jobs}"
cd "${source_dir}"
export ALLOW_MISSING_DEPENDENCIES=true
set +u
source build/envsetup.sh
lunch twrp_pissarro-eng
set -u
set -o pipefail
mka bootimage -j"${build_jobs}" 2>&1 | tee "${artifact_dir}/build.log"


In [ ]:
%%bash
set -Eeuo pipefail
source_dir=/content/twrp-pissarro
artifact_dir=/content/twrp-artifacts
image_path="${source_dir}/out/target/product/pissarro/boot.img"
test -s "${image_path}"
image_size="$(stat -c '%s' "${image_path}")"
image_magic="$(LC_ALL=C head -c 8 "${image_path}")"
[[ "${image_magic}" == 'ANDROID!' ]]
(( image_size >= 16777216 && image_size <= 134217728 ))
install -m 0644 "${image_path}" "${artifact_dir}/twrp-pissarro.img"
cd "${artifact_dir}"
sha256sum twrp-pissarro.img > twrp-pissarro.img.sha256
unpack_dir="$(mktemp -d)"
trap 'rm -rf "${unpack_dir}"' EXIT
unpack_output="$(python3 "${source_dir}/system/tools/mkbootimg/unpack_bootimg.py" \
  --boot_img twrp-pissarro.img --out "${unpack_dir}" --format=mkbootimg)"
printf '%s\n' "${unpack_output}" > twrp-pissarro.img.metadata.txt
for expected_value in \
  '--header_version 2' \
  '--pagesize 0x00000800' \
  '--kernel_offset 0x40080000' \
  '--ramdisk_offset 0x51100000' \
  '--tags_offset 0x47c80000' \
  '--dtb_offset 0x0000000047c80000' \
  'bootopt=64S3,32N2,64N2'; do
  grep -Fq -- "${expected_value}" <<< "${unpack_output}"
done
for component in kernel ramdisk dtb; do test -s "${unpack_dir}/${component}"; done
echo "镜像严格校验通过：${image_size} 字节"
cat twrp-pissarro.img.sha256


In [ ]:
# 下载镜像、校验信息、日志和固定版本清单。
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/twrp-pissarro-artifacts', 'zip', '/content/twrp-artifacts')
files.download(archive_path)


## 可选：复制到 Google Drive

仅在镜像已经校验通过后再执行下面的单元格。Drive 只保存最终压缩包，不参与源码同步或编译。

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copy2('/content/twrp-pissarro-artifacts.zip', '/content/drive/MyDrive/twrp-pissarro-artifacts.zip')
print('已复制到 Google Drive。')
